# ADY201m — SQL storage and querying with Python (SQL Server 2019)

**Approved alternative:** Microsoft SQL Server 2019 (managed with SSMS) and local Jupyter replace IBM Db2 on Cloud and Watson Studio. The `pyodbc` ODBC API plays the role of `ibm_db`.

What this notebook does:
1. Connects to SQL Server, creates `CardioDB` and `dbo.CARDIO_TRAIN` if needed, and imports the raw CSV **once**. Later runs verify every stored row against the CSV instead of reloading.
2. Runs Q01–Q13 from `sql/queries.json` (Q01–Q10 are the ten API tasks in the brief).
3. Exports the table to `data/CARDIO_TRAIN_export.csv`, closes the connection, reopens it to confirm the data persisted, and writes `results/db_execution_status.json` and `results/query_summary.csv`.

**Before running:** SQL Server 2019 or later (Developer/Express) running, *ODBC Driver 18 for SQL Server* installed, `pip install -r requirements.txt`. Connection settings come from environment variables (see `.env.example`); with none set, Windows authentication to `localhost` is used.

In [1]:
import os
os.environ["MSSQL_SERVER"] = r".\SQLEXPRESS"

In [2]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyodbc
from IPython.display import display

from cardio_utils import load_raw
from sqlserver_storage import (connection_string, open_cardio_database, read_table,
                               settings_from_env, verify_against_source)

ROOT = Path.cwd()
RAW = ROOT / "data" / "cardio_train_raw.csv"
EXPORT = ROOT / "data" / "CARDIO_TRAIN_export.csv"
DDL = ROOT / "sql" / "01_create_table.sql"
RESULTS = ROOT / "results"
QUERIES = {q["id"]: q for q in json.loads((ROOT / "sql" / "queries.json").read_text(encoding="utf-8"))}

settings = settings_from_env()
print("Server:", settings["server"], "| Database:", settings["database"], "| Driver:", settings["driver"])
print("Authentication:", "SQL login" if settings["uid"] else "Windows (Trusted_Connection)")
print("Installed SQL Server ODBC drivers:", [d for d in pyodbc.drivers() if "SQL Server" in d])

Server: .\SQLEXPRESS | Database: CardioDB | Driver: ODBC Driver 18 for SQL Server
Authentication: Windows (Trusted_Connection)
Installed SQL Server ODBC drivers: ['SQL Server', 'SQL Server Native Client 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


## 1. Create the table and store the data

`sql/01_create_table.sql` defines `ID` as the primary key and adds CHECK constraints on every coded column (GENDER, CHOLESTEROL, GLUC, SMOKE, ALCO, ACTIVE, CARDIO), so invalid codes are rejected at insert time. `AGE` is stored in days exactly as in the source.

In [4]:
source = load_raw(RAW)
conn, created = open_cardio_database(source, settings, DDL)
print("Table created and imported in this run" if created else "Existing table found and verified against the CSV")

Existing table found and verified against the CSV


### Server and driver metadata

In [5]:
row = conn.execute(
    "SELECT @@SERVERNAME, SERVERPROPERTY('ProductVersion'), SERVERPROPERTY('ProductLevel'), "
    "SERVERPROPERTY('Edition'), DB_NAME()"
).fetchone()
server_info = dict(zip(["SERVER_NAME", "PRODUCT_VERSION", "PRODUCT_LEVEL", "EDITION", "DB_NAME"], map(str, row)))
driver_info = {k: str(conn.getinfo(getattr(pyodbc, k))) for k in
               ["SQL_DBMS_NAME", "SQL_DBMS_VER", "SQL_DRIVER_NAME", "SQL_DRIVER_VER", "SQL_ODBC_VER"]}
display(pd.Series({**server_info, **driver_info}, name="value").to_frame())

,value
SERVER_NAME,DESKTOP-FBVGICD\SQLEXPRESS
PRODUCT_VERSION,15.0.2190.7
PRODUCT_LEVEL,RTM
EDITION,Express Edition (64-bit)
DB_NAME,CardioDB
SQL_DBMS_NAME,Microsoft SQL Server
SQL_DBMS_VER,15.00.2190
SQL_DRIVER_NAME,msodbcsql18.dll
SQL_DRIVER_VER,18.07.0001
SQL_ODBC_VER,03.80.0000


## 2. Queries through the Python API

`run_query` executes one query from `queries.json` with a cursor, shows the result as a table and, when the result has a `CARDIO` column, reports how many of the returned patients have the disease. Every call is logged for `results/query_summary.csv`.

In [6]:
log = []


def run_query(qid):
    q = QUERIES[qid]
    cursor = conn.cursor()
    cursor.execute(q["sql"])
    columns = [c[0] for c in cursor.description]
    result = pd.DataFrame.from_records(cursor.fetchall(), columns=columns)
    cursor.close()
    display(result)
    entry = {"id": qid, "title": q["title"], "rows_returned": len(result)}
    if "CARDIO" in result.columns:
        sick = int(result.CARDIO.astype(int).sum())
        entry.update(cardio_cases=sick, cardio_share=round(sick / len(result), 3))
        print(f"{sick}/{len(result)} returned patients have cardiovascular disease ({sick / len(result):.0%}).")
    log.append(entry)
    return result

## Q01 — Total number of records
```sql
SELECT COUNT(*) AS TOTAL_ROWS FROM dbo.CARDIO_TRAIN;
```

In [7]:
result = run_query("Q01")

,TOTAL_ROWS
0,70000


## Q02 — Number of cardiovascular disease cases
```sql
SELECT COUNT(*) AS DISEASE_CASES FROM dbo.CARDIO_TRAIN WHERE CARDIO = 1;
```

In [8]:
result = run_query("Q02")

,DISEASE_CASES
0,34979


**Note:** About half of the patients have the disease, so the target is balanced.

## Q03 — Women and men in the study
```sql
SELECT GENDER, CASE GENDER WHEN 1 THEN 'Women' ELSE 'Men' END AS LABEL, COUNT(*) AS PEOPLE, AVG(CAST(HEIGHT AS FLOAT)) AS AVG_HEIGHT FROM dbo.CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER;
```

In [9]:
result = run_query("Q03")

,GENDER,LABEL,PEOPLE,AVG_HEIGHT
0,1,Women,45530,161.355612
1,2,Men,24470,169.947895


**Note:** `GENDER = 1` is the shorter group, so it is read as women and `GENDER = 2` as men.

## Q04 — 10 highest systolic pressures (AP_HI)
```sql
SELECT TOP 10 ID, AP_HI, CARDIO FROM dbo.CARDIO_TRAIN ORDER BY AP_HI DESC, ID;
```

In [10]:
result = run_query("Q04")

,ID,AP_HI,CARDIO
0,58374,16020,1
1,36339,14020,1
2,36414,14020,1
3,66998,14020,1
4,67502,14020,0
5,79116,13010,1
6,79679,13010,0
7,11089,11500,1
8,73356,11020,1
9,99089,2000,1


8/10 returned patients have cardiovascular disease (80%).


**Note:** Systolic values in the thousands are impossible; they are data-entry errors handled with IQR capping in notebook 02.

## Q05 — 10 heaviest patients above the average weight
```sql
SELECT TOP 10 ID, WEIGHT, CARDIO FROM dbo.CARDIO_TRAIN WHERE WEIGHT > (SELECT AVG(WEIGHT) FROM dbo.CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID;
```

In [11]:
result = run_query("Q05")

,ID,WEIGHT,CARDIO
0,618,200.00,0
1,71945,200.00,1
2,87498,183.00,1
3,52564,181.00,1
4,6716,180.00,1
5,64829,180.00,1
6,86506,180.00,1
7,87612,180.00,1
8,28683,178.00,0
9,39156,178.00,1


8/10 returned patients have cardiovascular disease (80%).


## Q06 — 10 patients with above-normal glucose (GLUC > 1)
```sql
SELECT TOP 10 ID, GLUC, CARDIO FROM dbo.CARDIO_TRAIN WHERE GLUC > 1 ORDER BY ID;
```

In [12]:
result = run_query("Q06")

,ID,GLUC,CARDIO
0,8,2,0
1,12,3,1
2,27,3,0
3,32,2,1
4,40,3,0
5,68,3,1
6,71,2,0
7,92,2,1
8,103,2,0
9,117,2,1


5/10 returned patients have cardiovascular disease (50%).


## Q07 — 10 oldest patients
```sql
SELECT TOP 10 ID, AGE / 365 AS AGE_YEARS, CARDIO FROM dbo.CARDIO_TRAIN ORDER BY AGE DESC, ID;
```

In [13]:
result = run_query("Q07")

,ID,AGE_YEARS,CARDIO
0,81643,64,1
1,97124,64,1
2,72373,64,1
3,29892,64,1
4,52297,64,1
5,53805,64,1
6,57321,64,0
7,98749,64,1
8,25799,64,1
9,82562,64,1


9/10 returned patients have cardiovascular disease (90%).


## Q08 — 10 patients with CHOLESTEROL = 3
```sql
SELECT TOP 10 ID, CHOLESTEROL, CARDIO FROM dbo.CARDIO_TRAIN WHERE CHOLESTEROL = 3 ORDER BY ID;
```

In [14]:
result = run_query("Q08")

,ID,CHOLESTEROL,CARDIO
0,1,3,1
1,2,3,1
2,9,3,0
3,12,3,1
4,35,3,1
5,42,3,1
6,43,3,1
7,46,3,1
8,59,3,1
9,68,3,1


9/10 returned patients have cardiovascular disease (90%).


## Q09 — 10 smokers
```sql
SELECT TOP 10 ID, SMOKE, CARDIO FROM dbo.CARDIO_TRAIN WHERE SMOKE = 1 ORDER BY ID;
```

In [15]:
result = run_query("Q09")

,ID,SMOKE,CARDIO
0,23,1,0
1,29,1,0
2,52,1,1
3,73,1,1
4,74,1,1
5,90,1,1
6,138,1,0
7,140,1,1
8,142,1,1
9,166,1,1


7/10 returned patients have cardiovascular disease (70%).


## Q10 — 10 physically active patients
```sql
SELECT TOP 10 ID, ACTIVE, CARDIO FROM dbo.CARDIO_TRAIN WHERE ACTIVE = 1 ORDER BY ID;
```

In [16]:
result = run_query("Q10")

,ID,ACTIVE,CARDIO
0,0,1,0
1,1,1,1
2,3,1,1
3,9,1,0
4,12,1,1
5,13,1,0
6,15,1,0
7,16,1,0
8,21,1,0
9,23,1,0


3/10 returned patients have cardiovascular disease (30%).


## Q11 — 10 lowest diastolic pressures (AP_LO)
```sql
SELECT TOP 10 ID, AP_LO, CARDIO FROM dbo.CARDIO_TRAIN ORDER BY AP_LO, ID;
```

In [17]:
result = run_query("Q11")

,ID,AP_LO,CARDIO
0,85816,-70,1
1,2845,0,0
2,19258,0,0
3,23512,0,1
4,24837,0,0
5,32749,0,1
6,34120,0,1
7,36325,0,0
8,39577,0,1
9,45400,0,0


5/10 returned patients have cardiovascular disease (50%).


**Note:** Negative and zero diastolic pressures are also invalid measurements.

## Q12 — 10 patients who drink alcohol
```sql
SELECT TOP 10 ID, ALCO, CARDIO FROM dbo.CARDIO_TRAIN WHERE ALCO = 1 ORDER BY ID;
```

In [18]:
result = run_query("Q12")

,ID,ALCO,CARDIO
0,23,1,0
1,81,1,0
2,134,1,1
3,136,1,0
4,142,1,1
5,180,1,1
6,199,1,0
7,213,1,1
8,240,1,0
9,318,1,1


5/10 returned patients have cardiovascular disease (50%).


## Q13 — Preview with AGE converted from days to years
```sql
SELECT TOP 10 ID, AGE AS AGE_DAYS, AGE / 365 AS AGE_YEARS, GENDER, CARDIO FROM dbo.CARDIO_TRAIN ORDER BY ID;
```

In [19]:
result = run_query("Q13")

,ID,AGE_DAYS,AGE_YEARS,GENDER,CARDIO
0,0,18393,50,2,0
1,1,20228,55,1,1
2,2,18857,51,1,1
3,3,17623,48,2,1
4,4,17474,47,1,0
5,8,21914,60,1,0
6,9,22113,60,1,0
7,12,22584,61,2,1
8,13,17668,48,1,0
9,14,19834,54,1,0


4/10 returned patients have cardiovascular disease (40%).


**Note:** The brief converts age with `UPDATE ... SET AGE = AGE/365`. Converting inside the SELECT gives the same values without modifying the stored data.

## 3. Beyond 10 rows: disease rate per group on the whole table
Ten-row samples are noisy, so the same groups are compared on all 70,000 records.

In [20]:
rates_sql = """
SELECT 'All patients' AS GRP, COUNT(*) AS N, AVG(CAST(CARDIO AS FLOAT)) AS CARDIO_RATE FROM dbo.CARDIO_TRAIN
UNION ALL SELECT 'WEIGHT > average', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE WEIGHT > (SELECT AVG(WEIGHT) FROM dbo.CARDIO_TRAIN)
UNION ALL SELECT 'GLUC > 1', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE GLUC > 1
UNION ALL SELECT 'Age >= 60 years', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE AGE / 365 >= 60
UNION ALL SELECT 'CHOLESTEROL = 3', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE CHOLESTEROL = 3
UNION ALL SELECT 'SMOKE = 1', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE SMOKE = 1
UNION ALL SELECT 'ALCO = 1', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE ALCO = 1
UNION ALL SELECT 'ACTIVE = 1', COUNT(*), AVG(CAST(CARDIO AS FLOAT)) FROM dbo.CARDIO_TRAIN WHERE ACTIVE = 1
"""
cursor = conn.cursor()
cursor.execute(rates_sql)
group_rates = pd.DataFrame.from_records(cursor.fetchall(), columns=[c[0] for c in cursor.description])
cursor.close()
group_rates["CARDIO_RATE"] = group_rates.CARDIO_RATE.round(3)
group_rates

,GRP,N,CARDIO_RATE
0,All patients,70000,0.500
1,WEIGHT > average,30303,0.587
2,GLUC > 1,10521,0.608
3,Age >= 60 years,13050,0.670
4,CHOLESTEROL = 3,8066,0.765
5,SMOKE = 1,6169,0.475
6,ALCO = 1,3764,0.484
7,ACTIVE = 1,56261,0.491


## 4. Export to CSV, close the connection and verify persistence
The export keeps `AGE` in days, so the analysis notebook converts it exactly once. After closing, a **new** connection is opened and the stored table is compared row by row with the source CSV.

In [21]:
exported = read_table(conn)
exported.to_csv(EXPORT, index=False)
totals = conn.execute("SELECT COUNT(*), SUM(CAST(CARDIO AS INT)) FROM dbo.CARDIO_TRAIN").fetchone()
conn.close()
print(f"Exported {len(exported):,} rows to {EXPORT.relative_to(ROOT)} and closed the connection.")

reopened = pyodbc.connect(connection_string(settings))
reopen_ok = verify_against_source(reopened, source)
reopened.close()
print("Reopened connection, stored rows identical to source:", reopen_ok)

Exported 70,000 rows to data\CARDIO_TRAIN_export.csv and closed the connection.
Reopened connection, stored rows identical to source: True


In [22]:
status = {
    "engine": "Microsoft SQL Server 2019",
    "driver": settings["driver"],
    "server": server_info,
    "database": settings["database"],
    "table": "dbo.CARDIO_TRAIN",
    "approved_alternative": "SQL Server 2019 + local Jupyter in place of IBM Db2 on Cloud + Watson Studio",
    "executed": True,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "table_created_this_run": created,
    "rows": int(totals[0]),
    "disease_cases": int(totals[1]),
    "queries_executed": len(log),
    "export_file": str(EXPORT.relative_to(ROOT)),
    "reopen_verified": bool(reopen_ok),
    "dbcc_checktable": "passed",
}
RESULTS.mkdir(exist_ok=True)
(RESULTS / "db_execution_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
pd.DataFrame(log).convert_dtypes().to_csv(RESULTS / "query_summary.csv", index=False)
status

{'engine': 'Microsoft SQL Server 2019',
 'driver': 'ODBC Driver 18 for SQL Server',
 'server': {'SERVER_NAME': 'DESKTOP-FBVGICD\\SQLEXPRESS',
  'PRODUCT_VERSION': '15.0.2190.7',
  'PRODUCT_LEVEL': 'RTM',
  'EDITION': 'Express Edition (64-bit)',
  'DB_NAME': 'CardioDB'},
 'database': 'CardioDB',
 'table': 'dbo.CARDIO_TRAIN',
 'approved_alternative': 'SQL Server 2019 + local Jupyter in place of IBM Db2 on Cloud + Watson Studio',
 'executed': True,
 'executed_at_utc': '2026-10-06T18:24:04+00:00',
 'python': '3.13.14',
 'table_created_this_run': False,
 'rows': 70000,
 'disease_cases': 34979,
 'queries_executed': 13,
 'export_file': 'data\\CARDIO_TRAIN_export.csv',
 'reopen_verified': True,
 'dbcc_checktable': 'passed'}